In [2]:
import ollama
import json
import pandas as pd

In [3]:
# Load QULAC dataset from JSON file
# file contains queries, clarification questions, and metadata
with open(r"C:\Raina's Files\Semester IV - Thesis\Experiments\qulac.json") as f:
    data = json.load(f)

# convert to rows
df = pd.DataFrame(data)

print(df.head())

   topic_id  facet_id topic_facet_id topic_facet_question_id  \
0         1         1            1-1                   1-1-1   
1         1         1            1-1                  1-1-10   
2         1         1            1-1                  1-1-11   
3         1         1            1-1                  1-1-12   
4         1         1            1-1                   1-1-2   

               topic topic_type facet_type  \
0  obama family tree    faceted        nav   
1  obama family tree    faceted        nav   
2  obama family tree    faceted        nav   
3  obama family tree    faceted        nav   
4  obama family tree    faceted        nav   

                                          topic_desc  \
0  Find information on President Barack Obama\'s ...   
1  Find information on President Barack Obama\'s ...   
2  Find information on President Barack Obama\'s ...   
3  Find information on President Barack Obama\'s ...   
4  Find information on President Barack Obama\'s ...   

 

In [4]:
# unique queries
unique_queries = df[['topic_id', 'topic']].drop_duplicates()

sample_queries = unique_queries.sample(n=30, random_state=42)

sample_df = df[df['topic_id'].isin(sample_queries['topic_id'])]

# Reset index
sample_df = sample_df.reset_index(drop=True)
sample_df['ID'] = sample_df.index + 1

sample_df = sample_df[['ID', 'topic_id', 'topic', 'question']]

print(sample_df.head(10))

   ID  topic_id                topic  \
0   1       108  ralph owen brewster   
1   2       108  ralph owen brewster   
2   3       108  ralph owen brewster   
3   4       108  ralph owen brewster   
4   5       108  ralph owen brewster   
5   6       108  ralph owen brewster   
6   7       108  ralph owen brewster   
7   8       108  ralph owen brewster   
8   9       108  ralph owen brewster   
9  10       108  ralph owen brewster   

                                            question  
0  are you interested in learning more about ralp...  
1  would you like to know the political positions...  
2  would you like to know what movie star portray...  
3  would you like to know what political party ra...  
4                    how old is ralph owen brewseter  
5  may i provide you with information regarding w...  
6  may i tell you about brewsters parents where a...  
7              what is ralph owen brewser famous for  
8            what is ralph owen brewsters background  
9        

In [5]:
# drop empty question
sample_df = sample_df[sample_df['question'].notna()]
sample_df = sample_df[sample_df['question'].str.strip() != ""]

In [6]:
# BM25 to simulate retrieval
from rank_bm25 import BM25Okapi

# create a simple corpus
corpus = df[['topic', 'topic_desc']].drop_duplicates()['topic_desc'].dropna().tolist()

# tokenize
tokenized_corpus = [doc.lower().split() for doc in corpus]

# BM25 model
bm25 = BM25Okapi(tokenized_corpus)

def get_top_docs(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    docs = [corpus[i] for i in top_n]
    
    return docs if docs else []

In [7]:
# compute once per unique topic
topic_to_docs = {
    topic: get_top_docs(topic)
    for topic in sample_df['topic'].unique()
}

# map back
sample_df['retrieved_docs'] = sample_df['topic'].map(topic_to_docs)

print(sample_df[['topic', 'retrieved_docs']].head(3))

                 topic                                     retrieved_docs
0  ralph owen brewster  [Find biographical information about Ralph Owe...
1  ralph owen brewster  [Find biographical information about Ralph Owe...
2  ralph owen brewster  [Find biographical information about Ralph Owe...


In [8]:
# Generate answer using retrieved docs as context
def generate_answer(query, docs):
    context = "\n".join([f"Doc{i+1}: {doc}" for i, doc in enumerate(docs[:2])])[:500]

    prompt = f"""
    You are helping clarify an ambiguous search query.

    Based on the context, briefly state what the user might be looking for.

    Use ONE short sentence only.

    If unclear, say: Information is missing

    User query: {query}

    Context:
    {context}

    Answer:
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'num_predict': 80,
            'temperature': 0
        }
    )

    msg = response['message']
    answer = msg.get('content', '').strip()

    if not answer:
        answer = msg.get('thinking', '').strip()

    answer = answer.split("\n")[0].strip()

    # sanity check
    if not answer:
        return "Information is missing"

    return answer

In [9]:
# run on small sample
test_df = sample_df.head(3).copy()

answers = []
for _, row in test_df.iterrows():
    ans = generate_answer(row['topic'], row['retrieved_docs'])
    answers.append(ans)

test_df['answer'] = answers

In [10]:
pd.set_option('display.max_colwidth', None)
print(test_df[['topic', 'answer']])

                 topic  \
0  ralph owen brewster   
1  ralph owen brewster   
2  ralph owen brewster   

                                                                                               answer  
0  The user might be looking for information about Ralph Owen Brewster's biography or family history.  
1  The user might be looking for information about Ralph Owen Brewster's biography or family history.  
2  The user might be looking for information about Ralph Owen Brewster's biography or family history.  


In [11]:
# Baseline follow-up generation (plain prompting)

def generate_baseline_followup(query):
    prompt = f"""
    Generate ONE useful follow-up question to clarify the query.

    Rules:
    - Ask about a specific missing detail (e.g., location, type, features, time, etc.)
    - Do NOT ask vague questions like "What do you want to know?"
    - The question must narrow down the query and be relevant to the original query and potential information needs.
    - Frame the output as a question
    
    

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    msg = response['message']
    text = msg.get('content', '').strip()

    text = text.replace("Question:", "").replace("Follow-up", "").replace(":", "").strip()

    text = text.strip()

    # keep only first question sentence
    if "?" in text:
        text = text.split("?")[0] + "?"

    return text

In [12]:
# test
print(generate_baseline_followup("flushing"))
print(generate_baseline_followup("ontario california airport"))

What specific type of flushing are you referring to, such as toilet or plumbing system?
What specific features or services are available at Ontario International Airport in California?


In [13]:
# Aspect extraction
def extract_aspects(query):
    prompt = f"""
    Identify 3 specific aspects of the query.

    Rules:
    - Must be directly related to the query
    - Must be realistic information needs
    - Do NOT use generic aspects (e.g., definition, role, purpose)
    - Do NOT switch domain

    Return short phrases (2–5 words), one per line.

    Query: {query}

    Aspects:
    """
    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 80, 'temperature': 0}
    )

    # cleaning function
    def clean_aspects(text):
        lines = text.split("\n")
        cleaned = []

        for line in lines:
            line = line.strip()

            # remove numbering / labels
            line = line.replace("Output", "")
            line = line.replace(":", "")
            line = line.lstrip("0123456789. ").strip()

            # keep only meaningful phrases
            if len(line.split()) >= 2:
                cleaned.append(line)

        return cleaned[:3]

    raw = response['message']['content']
    return clean_aspects(raw)

In [14]:
# test aspect extraction
print(extract_aspects("ontario california airport"))
print("------")
print(extract_aspects("flushing"))
print("------")
print(extract_aspects("civil right movement"))

['Airport services', 'Airlines and destinations']
------
['Medical procedure', 'Plumbing technique', 'Biological waste management']
------
['Participants and leaders', 'Key events and protests', 'Legal battles and victories']


In [15]:
# Missing aspect identification
def find_missing_aspect(query, answer, aspects):
    prompt = f"""
    Query: {query}

    Aspects:
    {aspects}

    Answer:
    {answer}

    Task:
    From the list of aspects above, select ONE aspect that is NOT covered in the answer.

    If ALL aspects are already covered, return: NONE

    Rules:
    - You MUST choose ONLY from the provided aspects OR return NONE
    - Do NOT invent new aspects
    - Do NOT select aspects already clearly answered
    - Prefer the most important missing aspect
    - Output ONLY the exact aspect text OR NONE
    - Match wording exactly as given in aspects

    Missing aspect:
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 30, 'temperature': 0}
    )

    text = response['message']['content'].strip()

    # normalize
    clean_text = text.strip().lower()

    # STOP condition
    if clean_text == "none":
        return "NONE"

    # handle multiple aspects
    if "," in text:
        text = text.split(",")[0].strip()

    if " and " in text:
        text = text.split(" and ")[0].strip()

    # match with aspects
    for aspect in aspects:
        if aspect.lower() in text.lower():
            return aspect

    # fallback → STOP
    return "NONE"

In [16]:
# Improved follow-up generation

def generate_aspect_followup(query, missing_aspect):
    # handle fallback first
    if missing_aspect == "general information":
        return f"What specific details about {query} would help clarify your question?"
    
    prompt = f"""
    The user query is: {query}

    A key missing piece of information is: {missing_aspect}

    Generate ONE clear, direct question asking for this missing information.

    Rules:
    - The question MUST start with "What", "Which", "Who", "Where", or "When"
    - Do NOT use phrases like "Can you", "Could you", "Please", or "I"
    - Ask about a specific factual detail (NOT user preference)
    - Do NOT ask vague questions
    - Stay strictly within the topic
    - Keep it short and precise
    - Do NOT answer the query

    Output ONLY the question.
    """

    response = ollama.chat(
        model='deepseek-llm:7b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    
    text = response['message']['content'].strip()

    text = text.strip()

    if "?" in text:
        text = text.split("?")[0] + "?"

    return text

In [17]:
# test improved follow-up question generation
query = "ontario california airport"
docs = get_top_docs(query)

ans = generate_answer(query, docs)
aspects = extract_aspects(query)
missing = find_missing_aspect(query, ans, aspects)
imp_q = generate_aspect_followup(query, missing)

print("Aspects:", aspects)
print("Answer:", ans)
print("Improved:", imp_q)


Aspects: ['Airport services', 'Airlines and destinations']
Answer: The user might be looking for information about an airport in Ontario, California.
Improved: What airport is in Ontario, California?


In [30]:
# Iterative pipeline
def run_iterative_pipeline(query, steps=2):
    history = []
    context_query = query  # keep original

    for step in range(steps):
        docs = get_top_docs(context_query)
        answer = generate_answer(context_query, docs)

        aspects = extract_aspects(context_query)
        missing = find_missing_aspect(context_query, answer, aspects)

        if "," in missing:
            missing = missing.split(",")[0].strip()

        # initialize
        followup = None

        # store step FIRST
        history.append({
            "step": step + 1,
            "query": context_query,
            "answer": answer,
            "missing": missing,
            "followup": followup
        })

        # THEN stop
        if missing == "NONE":
            break

        followup = generate_aspect_followup(context_query, missing)

        # update followup
        history[-1]["followup"] = followup

        # combine queries
        context_query = followup

    return history

In [31]:
# test
result = run_iterative_pipeline("ontario california airport")

for step in result:
    print(step)
    print("-----")

{'step': 1, 'query': 'ontario california airport', 'answer': 'The user might be looking for information about an airport in Ontario, California.', 'missing': 'NONE', 'followup': None}
-----


In [32]:
# test on multiple queries
results = []

for query in sample_df['topic'].drop_duplicates().head(3):
    res = run_iterative_pipeline(query)
    results.append(res)

In [33]:
for i, res in enumerate(results):
    print(f"Query {i+1}")
    for step in res:
        print(step)
    print("------")

Query 1
{'step': 1, 'query': 'ralph owen brewster', 'answer': "The user might be looking for information about Ralph Owen Brewster's biography or family history.", 'missing': 'Role in the development of public education', 'followup': 'What role did Ralph Owen Brewster play in the development of public education?'}
{'step': 2, 'query': 'What role did Ralph Owen Brewster play in the development of public education?', 'answer': 'Ralph Owen Brewster was a prominent figure in the development of public education, particularly through his work on curriculum reform and educational administration.', 'missing': 'Founder of Boston Public Schools', 'followup': 'What role did Ralph Owen Brewster play in the development of public education, specifically as the founder of Boston Public Schools?'}
------
Query 2
{'step': 1, 'query': 'hp mini 2140', 'answer': 'The user might be looking for information about the HP Mini 2140 laptop computer.', 'missing': 'NONE', 'followup': None}
------
Query 3
{'step':

In [ ]:
# Evaluation using BERTScore
from bert_score import score

def compute_bertscore(preds, refs):
    P, R, F1 = score(preds, refs, lang="en", verbose=False)
    return F1.mean().item()

In [ ]:
# use ONE reference per query
test_df = unique_queries.head(5).copy()

test_df['Baseline'] = test_df['topic'].apply(generate_baseline_followup)

test_df['Proposed'] = test_df['topic'].apply(
    lambda q: generate_aspect_followup(
        q,
        find_missing_aspect(
            q,
            generate_answer(q, get_top_docs(q)),
            extract_aspects(q)
        )
    )
)

print(test_df[['topic', 'Baseline', 'Proposed']])

                           topic  \
0              obama family tree   
39                cheap internet   
129  ritz carlton lake las vegas   
189            fickle creek farm   
234              madam cj walker   

                                                                                                                         Baseline  \
0    What specific details about the Obama family tree are you looking for, such as ancestors or relationships within the family?   
39                                                             The most cost-effective internet service provider in my area is...   
129                                                         The Ritz Carlton Lake Las Vegas is located on which side of the lake?   
189                                                      What type of activities or produce does Fickle Creek Farm specialize in?   
234                    Who was Madam C.J. Walker, and what were her contributions in the field of hair care and entrep

In [ ]:
# compute BERTScore
full_df = unique_queries.copy()

full_df['Baseline'] = full_df['topic'].apply(generate_baseline_followup)

full_df['Proposed'] = full_df['topic'].apply(
    lambda q: generate_aspect_followup(
        q,
        find_missing_aspect(
            q,
            generate_answer(q, get_top_docs(q)),
            extract_aspects(q)
        )
    )
)

In [122]:
# attach one reference question per topic
ref_df = df.drop_duplicates(subset=['topic_id'])[['topic_id', 'question']]

final_df = full_df.merge(ref_df, on='topic_id')

In [123]:
# remove any broken rows
final_df = final_df.dropna(subset=['Baseline', 'Proposed', 'question'])

print(len(final_df))  # sanity check (~198)

198


In [ ]:
# compute BERTScore
from bert_score import score

def compute_bertscore(preds, refs):
    P, R, F1 = score(preds, refs, lang="en", verbose=False)
    return F1.mean().item()

refs = final_df['question'].tolist()

baseline_score = compute_bertscore(final_df['Baseline'].tolist(), refs)
proposed_score = compute_bertscore(final_df['Proposed'].tolist(), refs)

print("Baseline:", baseline_score)
print("Proposed:", proposed_score)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1583.67it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2616.56it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status    

Baseline: 0.8608163595199585
Proposed: 0.8653765320777893


In [125]:
final_df.to_csv("final_results_198.csv", index=False)

In [127]:
# comparison table

# sample queries
sample_compare = full_df.sample(n=10, random_state=42).copy()

# generate iterative
iter1_list = []
iter2_list = []

for query in sample_compare['topic']:
    res = run_iterative_pipeline(query)

    iter1_list.append(res[0]['followup'])
    iter2_list.append(res[1]['followup'])

sample_compare['Iter-1'] = iter1_list
sample_compare['Iter-2'] = iter2_list

In [128]:
final_table = sample_compare[['topic', 'Baseline', 'Proposed', 'Iter-1', 'Iter-2']]

pd.set_option('display.max_colwidth', None)
print(final_table)

                                        topic  \
3262                              porterville   
6040                              rick warren   
864                       adobe indian houses   
7648                          indexed annuity   
8541                                 bellevue   
6823                       elliptical trainer   
7606                alexian brothers hospital   
1515                  dutchess county tourism   
983            california franchise tax board   
9205  to be or not to be that is the question   

                                                                                                                                                                     Baseline  \
3262                                                                                                                                            Where is Porterville located?   
6040                                                                                                Who is

In [129]:
final_table.to_csv("comparison_table.csv", index=False)